# Ensemble Semillas y Modelos

### 9.1 Objetivo

Este notebook tiene como objetivo consolidar las predicciones generadas por múltiples ejecuciones y experimentos.

Para ello, realiza el siguiente flujo:

Lee los archivos prediccion_semilla_<nro_semilla>.txt pertenecientes a cada experimento desde sus respectivas carpetas (WF<nro_experimento>_multi_semilla) dentro del bucket.

Convierte la probabilidad de baja de cada semilla-experimento a una escala de ranking percentil continuo, dividiendo la posición ordinal del cliente por la cantidad total de clientes.

Consolida todas las corridas calculando el promedio de dicho score por cliente a través de todas las semillas y experimentos.

Genera un dataset final con las columnas numero_de_cliente y prob (que almacena el ranking promedio).

Envío a Kaggle:

Finalmente, ordena el dataset consolidado en forma descendente por la columna prob, asigna el estímulo (Predicted = 1) a los primeros clientes según distintos cortes de envíos y realiza las entregas automáticas a Kaggle.

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")
if( !require("stringr")) install.packages("stringr")

if( !require("dplyr")) install.packages("dplyr")

#### Parametros

In [ ]:
PARAM <- list()
PARAM$experimento <- "9121_ensemble"


#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/"))

### 9.2   Leo los archivos de predicciones y calculo la nueva probabilidad

In [ ]:
# 1. Definir los dos experimentos que querés combinar
# EXP 1: WF9100_multi_semilla (undersampling 0.1)
# EXP 2: WF9104_multi_semilla (undersampling 0.01)
experimentos <- c("WF9100_multi_semilla", "WF9104_multi_semilla") 
base_path <- "/content/buckets/b1/exp"

lista_datasets <- list()

# 2. Recorrer cada experimento
for (exp in experimentos) {
  exp_dir <- file.path(base_path, exp)
  
  if (!dir.exists(exp_dir)) {
    warning(paste("No existe el directorio para el experimento:", exp_dir))
    next
  }
  
  # Buscar todos los archivos prediccion_semilla_*.txt
  archivos_prediccion <- list.files(
    path = exp_dir, 
    pattern = "^prediccion_semilla_.*\\.txt$", 
    full.names = TRUE
  )
  
  cat("\n=== Procesando experimento:", exp, "(", length(archivos_prediccion), "archivos ) ===\n")
  
  for (archivo in archivos_prediccion) {
    # Lectura rápida con data.table
    dt <- fread(archivo)
    
    nombre_archivo <- basename(archivo)
    num_semilla <- sub("^prediccion_semilla_(.*)\\.txt$", "\\1", nombre_archivo)
    
    # ----------------------------------------------------------------------
    # CONVERSIÓN A RANKING PERCENTIL (ENTRE 0 Y 1)
    # ----------------------------------------------------------------------
    # frank() calcula la posición en el ranking de mayor a menor o menor a mayor.
    # Al dividir por .N (total de filas), transformamos la probabilidad en un percentil.
    dt[, rank_norm := frank(prob, ties.method = "average") / .N]
    
    dt[, Exp := exp]
    dt[, Semilla := num_semilla]
    
    lista_datasets[[length(lista_datasets) + 1]] <- dt
    cat("  - Leído y transformado a percentil:", nombre_archivo, "\n")
  }
}

# Unir todas las tablas en un data.table consolidado
dataset_completo <- rbindlist(lista_datasets)

# ----------------------------------------------------------------------
# PROMEDIO DE RANKINGS (ENSEMBLE FINAL)
# ----------------------------------------------------------------------
# Promediamos rank_norm en lugar de prob por cliente
tb_prediccion <- dataset_completo[, .(prob = mean(rank_norm, na.rm = TRUE)), by = numero_de_cliente]

cat("\nTotal clientes consolidados en ensemble de rankings:", nrow(tb_prediccion), "\n")

# Guardar en la carpeta del experimento actual
setwd(file.path("/content/buckets/b1/exp", experimento_folder))

fwrite(
  tb_prediccion,
  file = "prediccion.txt",
  sep = "\t"
)

### 9.3 Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# leo archivo con las predicciones promedio finales
tb_prediccion <- fread(paste0("prediccion.txt"))

PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1850, 1950, by = 100)
nombre_ensemble <- paste0(str_replace(str_replace(experimentos,"_multi_semilla",""),"WF",""),collapse = "_")

PARAM$modelo <- paste0("ensemble_",nombre_ensemble)
  
# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=ensemble_", nombre_ensemble,"_10semillas",
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")